In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim import AdamW, Muon

import matplotlib.pyplot as plt
import torchvision

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from pytorch_katas.settings import DATA_DIR

In [3]:
train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761]),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761]),
    ]
)

train_dataset = datasets.CIFAR100(root=DATA_DIR, train=True, transform=train_transform, download=True)
test_dataset = datasets.CIFAR100(root=DATA_DIR, train=False, transform=val_transform, download=True)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")

/home/felix/repos/pytorch-katas/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000, Test: 10000
Number of classes: 100


In [69]:
import math


class Tokenizer(nn.Module):
    def __init__(self, patch_size: int = 4, dim: int = 32, n_token: int = 64):
        super().__init__()

        self.dim = dim
        self.conv = nn.Conv2d(kernel_size=patch_size, in_channels=3, out_channels=dim, stride=patch_size)
        self.pos_embedding = nn.Parameter(torch.randn(1, n_token, dim) * 0.02)  # initialization

    def forward(self, x_bchw: torch.Tensor) -> torch.Tensor:
        x = self.conv(x_bchw)
        _, _, h, w = x.shape
        x_bsd = torch.reshape(x, (-1, h * w, self.dim))
        return x_bsd + self.pos_embedding


class MHA(nn.Module):
    def __init__(self, dim: int = 192, n_heads: int = 3):
        super().__init__()
        self.dim = dim
        self.n_heads = n_heads

        self.proj = nn.Linear(dim, dim * 3, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x_bsd: torch.Tensor) -> torch.Tensor:
        b, s, _ = x_bsd.shape

        x_bs3d = self.proj(x_bsd)
        q_bsd, k_bsd, v_bsd = x_bs3d.chunk(3, -1)

        q_bhsd = q_bsd.reshape((b, s, self.n_heads, -1)).permute((0, 2, 1, 3))
        k_bhds = k_bsd.reshape((b, s, self.n_heads, -1)).permute(0, 2, 3, 1)
        v_bhsd = v_bsd.reshape((b, s, self.n_heads, -1)).permute(0, 2, 1, 3)

        attn_bhss = F.softmax(torch.einsum("b h p d, b h d q -> b h p q", q_bhsd, k_bhds) / math.sqrt(self.dim // self.n_heads), dim=-1)
        attn_bhsd = torch.einsum("b h p q, b h q d -> b h p d", attn_bhss, v_bhsd)

        attn_bsd = torch.reshape(attn_bhsd, (b, -1, self.dim))
        out_bsd = self.out_proj(attn_bsd)

        return out_bsd


class TransformerBlock(nn.Module):
    def __init__(self, dim: int = 192, n_heads: int = 3, exp_factor: int = 4):
        super().__init__()
        self.mha = MHA(dim=dim, n_heads=n_heads)

        self.norm1 = nn.LayerNorm(normalized_shape=dim)
        self.norm2 = nn.LayerNorm(normalized_shape=dim)

        self.expansion = nn.Sequential(nn.Linear(dim, dim * exp_factor), nn.GELU(), nn.Linear(dim * exp_factor, dim))

    def forward(self, x_bsd: torch.Tensor):
        x_norm_bsd = self.norm1(x_bsd)
        x_bsd = x_bsd + self.mha(x_norm_bsd)

        x_norm_bsd = self.norm2(x_bsd)
        x_bsd = x_bsd + self.expansion(x_norm_bsd)

        return x_bsd


class Transformer(nn.Module):
    def __init__(
        self,
        dim: int = 192,
        n_heads: int = 3,
        exp_factor: int = 4,
        n_layers: int = 12,
        patch_size: int = 4,
        n_token: int = 64,
        n_classes: int = 100
    ):
        super().__init__()

        self.tokenizer = Tokenizer(patch_size=patch_size, dim=dim, n_token=n_token)
        self.layers = nn.ModuleList(TransformerBlock(dim=dim, n_heads=n_heads, exp_factor=exp_factor) for i in range(n_layers))
        self.final_norm = nn.LayerNorm(dim)
        self.classifier = nn.Linear(dim, n_classes)

    def forward(self, x_bchw: torch.Tensor) -> torch.Tensor:
        x_bsd = self.tokenizer(x_bchw)
        for layer in self.layers:
            x_bsd = layer(x_bsd)
        x_bsd = self.final_norm(x_bsd)
        mean = x_bsd.mean(dim=1)
        return self.classifier(mean)

In [79]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

m = Transformer()
m = m.to(device)

In [80]:
from torch.optim.lr_scheduler import OneCycleLR

from tqdm.notebook import tqdm

n_epochs = 10
bs = 512

train_dl = DataLoader(train_dataset, batch_size=bs, shuffle=True, num_workers=8)
test_dl = DataLoader(test_dataset, batch_size=bs, shuffle=False, num_workers=8)

optimizer = AdamW(m.parameters(), lr=1e-3)
scheduler = OneCycleLR(
    optimizer,
    max_lr=1e-3,
    epochs=n_epochs,
    steps_per_epoch=len(train_dl),
    pct_start=0.1,
    div_factor=10,
    final_div_factor=100,
)

for epoch in range(n_epochs):
    train_pbar = tqdm(train_dl)
    m.train()
    for x, y in train_pbar:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss = F.cross_entropy(m(x), y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_pbar.set_postfix(loss=loss.item(), lr=f"{scheduler.get_last_lr()[0]:.2e}")

    val_pbar = tqdm(test_dl)
    m.eval()
    total = 0
    correct = 0
    for x, y in val_pbar:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        pred = m(x)
        loss = F.cross_entropy(pred, y)
        total += y.size(0)
        correct += (pred.argmax(dim=1) == y).sum().item()
        val_pbar.set_postfix(loss=loss.item())

    print(f"Accuracy: {correct / total:.3f}")

  0%|          | 0/98 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Accuracy: 0.101


  0%|          | 0/98 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/20 [00:00<?, ?it/s]

Accuracy: 0.317


  0%|          | 0/98 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Accuracy: 0.316
